# N2 — Financial Ratio Normalization & Feature Selection

Loads `panel_clean.parquet`, identifies all engineered financial ratios, runs correlation-based
feature pruning (threshold = 0.90), winsorises per fyear, z-normalises per fyear, imputes
remaining NaN with 0, and saves `financials_normalized.parquet`.

**Output fed into:** N6 (kNN financial peer lists → M1)


In [ ]:
# imports & config
import sys
from pathlib import Path

notebook_dir = Path('__file__').parent if '__file__' in dir() else Path.cwd()
repo_root = next(
    (p for p in [notebook_dir, *notebook_dir.parents] if (p / 'config.py').exists()),
    None
)
if repo_root is None:
    raise FileNotFoundError('config.py not found — check repo structure')
sys.path.insert(0, str(repo_root))

from config import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from scipy import stats

print("Config loaded.")
print(f"  PANEL_CLEAN      : {PANEL_CLEAN}")
print(f"  FINANCIALS_NORM  : {FINANCIALS_NORM}")
print(f"  WINSOR_BOUNDS    : {WINSOR_BOUNDS}")
print(f"  CORR THRESHOLD   : {CORRELATION_THRESHOLD}")


In [ ]:
INPUTS  = [PANEL_CLEAN]
OUTPUTS = [FINANCIALS_NORM, DATA_RESULTS / "selected_features.json"]
# PURPOSE : Correlation filtering, winsorisation, z-normalisation of financial ratios
# RUNTIME : ~3 min
# DEPENDS : panel_clean.parquet (N1)


## 1. Load Panel

In [ ]:
# load panel_clean
df = pd.read_parquet(PANEL_CLEAN)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Years : {sorted(df['fyear'].unique())}")
print(f"Firms : {df['gvkey'].nunique():,} unique GVKEYs")


## 2. Identify Financial Ratio Columns

We select only engineered ratio columns — those ending in `_wr` (Worldscope-style, winsorised)
or `_an` (anomaly-style). Raw Compustat items, lag/delta columns, identifiers, and target
multiples are explicitly excluded.


In [ ]:
# identify candidate ratio columns
# Include: engineered ratios (_wr, _an suffix)
# Exclude: raw items, lags (l_*), deltas (d_*), identifiers, targets

EXCLUDE_ALWAYS = {
    'gvkey', 'tic', 'conm', 'sic', 'fyear', 'datadate', 'curcd',
    'ff49_num', 'ff49_abbr', 'industry',
    'market_cap', 'ev', 'ln_m2b', 'ln_v2a', 'ln_v2s',
    'assets', 'book', 'sale_ac', 'CAPMbeta',
    'be', 'ocf', 'ibadj',
    'prcc_f', 'csho',
}

candidate_cols = [
    c for c in df.columns
    if (c.endswith('_wr') or c.endswith('_an'))
    and not c.startswith('l_')
    and not c.startswith('d_')
    and c not in EXCLUDE_ALWAYS
]

print(f"Candidate ratio columns: {len(candidate_cols)}")
print("\nSample (first 20):")
for c in candidate_cols[:20]:
    print(f"  {c}")
print(f"  ... and {len(candidate_cols)-20} more")


## 3. Coverage Check

Drop any ratio with >80% missing values across the full sample — these add noise, not signal.

In [ ]:
# drop low-coverage ratios (>80% NaN)
coverage = df[candidate_cols].isna().mean().sort_values(ascending=False)
drop_coverage = coverage[coverage > MISSING_THRESHOLD].index.tolist()

print(f"Ratios with >{MISSING_THRESHOLD*100:.0f}% missing → DROP: {len(drop_coverage)}")
for c in drop_coverage:
    print(f"  {c:<35} {coverage[c]*100:.1f}% missing")

candidate_cols = [c for c in candidate_cols if c not in drop_coverage]
print(f"\nRemaining after coverage filter: {len(candidate_cols)}")


## 4. Correlation-Based Feature Pruning

Compute the Spearman correlation matrix across all candidate ratios (pooled across years).
For any pair exceeding the 0.90 threshold, drop the feature with the **higher overall
missing-value rate** — retaining the more complete variable. This follows standard practice
in cross-sectional asset pricing (e.g. Geertsema & Lu 2023).


In [ ]:
# compute Spearman correlation matrix
# Use Spearman (rank-based) — robust to outliers before winsorisation
print("Computing Spearman correlation matrix...")
print(f"  Matrix size: {len(candidate_cols)} × {len(candidate_cols)}")

corr_matrix = df[candidate_cols].corr(method='spearman')
print("Done.")


In [ ]:
# visualise correlation heatmap (full matrix)
fig, ax = plt.subplots(figsize=(18, 15))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap='RdBu_r',
    vmin=-1, vmax=1,
    center=0,
    square=True,
    linewidths=0,
    cbar_kws={"shrink": 0.5},
    ax=ax,
    xticklabels=False,
    yticklabels=False,
)
ax.set_title(f"Spearman Correlation Matrix — {len(candidate_cols)} candidate ratios", fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES / "n2_correlation_heatmap_full.pdf", dpi=FIGURE_DPI)
plt.show()
print(f"Saved: {FIGURES / 'n2_correlation_heatmap_full.pdf'}")


In [ ]:
# identify highly correlated pairs (|r| > CORRELATION_THRESHOLD)
# Work on upper triangle only
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={'level_0': 'feature_a', 'level_1': 'feature_b', 0: 'spearman_r'})
    .query(f"abs(spearman_r) > {CORRELATION_THRESHOLD}")
    .sort_values('spearman_r', ascending=False)
    .reset_index(drop=True)
)

print(f"Pairs with |Spearman r| > {CORRELATION_THRESHOLD}: {len(high_corr_pairs)}")
print()
print(high_corr_pairs.to_string(index=False))


In [ ]:
# greedy correlation pruning
# For each correlated pair: drop the feature with higher missing-value rate
# Ties broken by dropping feature_b (arbitrary but consistent)

missing_rate = df[candidate_cols].isna().mean()
to_drop = set()

# Sort pairs by |r| descending so we handle strongest correlations first
pairs_sorted = high_corr_pairs.reindex(
    high_corr_pairs['spearman_r'].abs().sort_values(ascending=False).index
)

for _, row in pairs_sorted.iterrows():
    a, b = row['feature_a'], row['feature_b']
    if a in to_drop or b in to_drop:
        continue   
    # Drop the one with more missing data; tie → drop b
    if missing_rate[a] >= missing_rate[b]:
        to_drop.add(a)
        kept, dropped = b, a
    else:
        to_drop.add(b)
        kept, dropped = a, b

print(f"Features dropped by correlation pruning: {len(to_drop)}")
print()
print(f"{'Dropped':<35} {'Kept':<35} {'|r|'}")
print("-" * 80)
for _, row in pairs_sorted.iterrows():
    a, b = row['feature_a'], row['feature_b']
    if a in to_drop and b not in to_drop:
        print(f"{a:<35} {b:<35} {abs(row['spearman_r']):.3f}")
    elif b in to_drop and a not in to_drop:
        print(f"{b:<35} {a:<35} {abs(row['spearman_r']):.3f}")

selected_cols = [c for c in candidate_cols if c not in to_drop]
print(f"\nFeatures remaining after pruning: {len(selected_cols)}")


In [ ]:
# visualise reduced correlation matrix
fig, ax = plt.subplots(figsize=(16, 13))
corr_reduced = df[selected_cols].corr(method='spearman')
mask = np.triu(np.ones_like(corr_reduced, dtype=bool))
sns.heatmap(
    corr_reduced,
    mask=mask,
    cmap='RdBu_r',
    vmin=-1, vmax=1,
    center=0,
    square=True,
    linewidths=0,
    cbar_kws={"shrink": 0.5},
    ax=ax,
    xticklabels=False,
    yticklabels=False,
)
ax.set_title(
    f"Spearman Correlation Matrix after Pruning — {len(selected_cols)} features "
    f"(threshold = {CORRELATION_THRESHOLD})",
    fontsize=13
)
plt.tight_layout()
plt.savefig(FIGURES / "n2_correlation_heatmap_pruned.pdf", dpi=FIGURE_DPI)
plt.show()
print(f"Saved: {FIGURES / 'n2_correlation_heatmap_pruned.pdf'}")
print(f"Max off-diagonal |r| after pruning: {corr_reduced.where(mask==False).abs().max().max():.3f}")


## 5. Save Selected Feature List

Persist the final feature list to JSON so N6 and beyond consume exactly the same set.

In [ ]:
# save selected features to JSON
selected_features_path = SELECTED_FEATURES_FILE

feature_manifest = {
    "n_candidates"         : len(candidate_cols),
    "n_dropped_coverage"   : len(drop_coverage),
    "n_dropped_correlation": len(to_drop),
    "n_selected"           : len(selected_cols),
    "corr_threshold"       : CORRELATION_THRESHOLD,
    "missing_threshold"    : MISSING_THRESHOLD,
    "dropped_coverage"     : sorted(drop_coverage),
    "dropped_correlation"  : sorted(list(to_drop)),
    "selected_features"    : selected_cols,
}

with open(selected_features_path, 'w') as f:
    json.dump(feature_manifest, f, indent=2)

print(f"Saved: {selected_features_path}")
print(f"\nFeature selection summary:")
print(f"  Candidates identified : {len(candidate_cols) + len(drop_coverage)}")
print(f"  Dropped (coverage)    : {len(drop_coverage)}")
print(f"  Dropped (correlation) : {len(to_drop)}")
print(f"  Final feature count   : {len(selected_cols)}")


## 6. Winsorisation

Applied **per fyear cross-section** at the 1st and 99th percentile (config: `WINSOR_BOUNDS`).
This follows Geertsema & Lu (2023) and removes the influence of extreme outliers before
normalisation without discarding observations.

> **Note:** winsorisation is applied to input *features* only. Target multiples
> (`ln_v2s`, `ln_v2a`, `ln_m2b`) are winsorised separately in N4.


In [ ]:
# winsorise selected features per fyear
print(f"Winsorising {len(selected_cols)} features at {WINSOR_BOUNDS} per fyear...")

df_work = df[['gvkey', 'tic', 'fyear', 'ff49_num', 'ff49_abbr', 'industry'] + selected_cols].copy()

lo, hi = WINSOR_BOUNDS

for yr in YEARS:
    mask = df_work['fyear'] == yr
    for col in selected_cols:
        lo_val = df_work.loc[mask, col].quantile(lo)
        hi_val = df_work.loc[mask, col].quantile(hi)
        df_work.loc[mask, col] = df_work.loc[mask, col].clip(lower=lo_val, upper=hi_val)

print("Winsorisation complete.")

# Spot check
sample_col = selected_cols[0]
print(f"\nSpot check — {sample_col}:")
for yr in YEARS:
    yr_data = df_work[df_work['fyear'] == yr][sample_col]
    print(f"  {yr}: min={yr_data.min():.3f}  max={yr_data.max():.3f}  NaN={yr_data.isna().sum()}")

## 7. Z-Normalisation

Standardise each feature to zero mean and unit variance **within each fyear cross-section**.
This ensures the kNN distance metric (cosine) is not dominated by features with large
absolute scales, and makes features comparable across years.


In [ ]:
# z-normalise per fyear
print(f"Z-normalising {len(selected_cols)} features per fyear...")

for yr in YEARS:
    mask = df_work['fyear'] == yr
    for col in selected_cols:
        mu  = df_work.loc[mask, col].mean()
        std = df_work.loc[mask, col].std()
        if std > 0:
            df_work.loc[mask, col] = (df_work.loc[mask, col] - mu) / std
        else:
            df_work.loc[mask, col] = 0.0

print("Z-normalisation complete.")

# Verify: mean ≈ 0, std ≈ 1 per year per feature (spot check)
print("\nNormalisation check (mean / std per year, averaged across features):")
for yr in YEARS:
    yr_data = df_work[df_work['fyear'] == yr][selected_cols]
    print(f"  {yr}: mean={yr_data.mean().mean():.4f}  std={yr_data.std().mean():.4f}")

## 8. NaN Imputation

After z-normalisation, remaining NaN values (from structural missing data — e.g. inventory
turnover for firms with no inventory) are filled with **0**, which equals the cross-sectional
mean of the normalised distribution. This is the standard academic treatment (e.g. Hou, Xue
& Zhang 2020; Geertsema & Lu 2023) as it is neutral with respect to the similarity metric
and avoids listwise deletion.


In [ ]:
# impute remaining NaN with 0 (cross-sectional mean post z-score)
nan_before = df_work[selected_cols].isna().sum().sum()
df_work[selected_cols] = df_work[selected_cols].fillna(0.0)
nan_after  = df_work[selected_cols].isna().sum().sum()

print(f"NaN before imputation : {nan_before:,}")
print(f"NaN after imputation  : {nan_after:,}")
print()

# Per-feature NaN rate before imputation (informational)
nan_by_feature = (
    df[selected_cols].isna().mean()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'index': 'feature', 0: 'nan_rate'})
    .head(20)
)
print("Top 20 features by pre-imputation NaN rate:")
print(nan_by_feature.to_string(index=False))


## 9. Distribution Check

Spot-check normalised distributions for a sample of 12 features — confirms winsorisation
and z-normalisation behaved as expected before writing output.

In [ ]:
# distribution plots for 12 sample features
sample_features = selected_cols[:12]
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(sample_features):
    axes[i].hist(df_work[col], bins=60, color='steelblue', alpha=0.7, edgecolor='none')
    axes[i].axvline(0, color='red', linewidth=0.8, linestyle='--')
    axes[i].set_title(col, fontsize=8)
    axes[i].set_xlabel('z-score', fontsize=7)
    axes[i].tick_params(labelsize=7)

plt.suptitle("Normalised Feature Distributions (sample of 12)", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / "n2_feature_distributions.pdf", dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"Saved: {FIGURES / 'n2_feature_distributions.pdf'}")


## 10. Save Normalized Output

In [ ]:
# save financials_normalized.parquet
# Schema: gvkey | tic | fyear | ff49_num | ff49_abbr | industry | <selected_cols>
FINANCIALS_NORM.parent.mkdir(parents=True, exist_ok=True)
df_work.to_parquet(FINANCIALS_NORM, index=False)

print(f"Saved : {FINANCIALS_NORM}")
print(f"Shape : {df_work.shape[0]:,} rows × {df_work.shape[1]} columns")
print(f"  Identifier cols : 6 (gvkey, tic, fyear, ff49_num, ff49_abbr, industry)")
print(f"  Feature cols    : {len(selected_cols)}")
print(f"  Years           : {sorted(df_work['fyear'].unique())}")
print()
print(f"Firm-years per year:")
print(df_work.groupby('fyear')['gvkey'].count().to_string())


In [ ]:
# final summary
print("=" * 60)
print("N2 COMPLETE — FINANCIAL NORMALISATION SUMMARY")
print("=" * 60)
print(f"  Input  (panel_clean)      : {df.shape[0]:,} firm-years")
print(f"  Output (financials_norm)  : {df_work.shape[0]:,} firm-years")
print()
print(f"  Feature pipeline:")
print(f"    Candidate ratios        : {len(candidate_cols) + len(drop_coverage)}")
print(f"    Dropped — coverage      : {len(drop_coverage)}  (>{MISSING_THRESHOLD*100:.0f}% NaN)")
print(f"    Dropped — correlation   : {len(to_drop)}  (Spearman |r| > {CORRELATION_THRESHOLD})")
print(f"    Final feature count     : {len(selected_cols)}")
print()
print(f"  Transformations applied (in order):")
print(f"    1. Winsorisation        : {WINSOR_BOUNDS} per fyear cross-section")
print(f"    2. Z-normalisation      : mean=0, std=1 per fyear cross-section")
print(f"    3. NaN imputation       : fill with 0 (= cross-sectional mean)")
print()
print(f"  Outputs written:")
print(f"    {FINANCIALS_NORM}")
print(f"    {SELECTED_FEATURES_FILE}")
print()
print("  Next: N3 (text pipeline), N4 (multiples), N5 (M0), N6 (M1 kNN)")
print("        — all parallelisable now that N1 + N2 are complete")
